In [1]:
import sys
import polars as pl
sys.path.insert(0, '..')
import time
import json
from fs_thesis import sql, show
from sklearn.metrics import classification_report, confusion_matrix
import plotly.express as px
import plotly.figure_factory as ff
import plotly.graph_objects as go
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime
import warnings
from tqdm.auto import tqdm
from sklearn.metrics import (recall_score, precision_score, f1_score, 
                             accuracy_score, roc_auc_score, confusion_matrix)


In [2]:
# ── Run-Ordner (einmalig für das ganze Notebook) ──
_run_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_DIR = Path(f"/Users/andrey/Repositories/fs-thesis/models/runs/tab_pfn_{_run_timestamp}")
PLOTS_DIR = RUN_DIR / "plots"
RESULTS_DIR = RUN_DIR / "results"
for d in [PLOTS_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

_plot_counter = 0

def show_and_save(fig, name: str = None, width=1600, height=600):
    """fig.show() + PNG speichern in den aktuellen Run-Ordner."""
    global _plot_counter
    _plot_counter += 1
    filename = name or f"plot_{_plot_counter:02d}"
    path = PLOTS_DIR / f"{filename}.png"
    fig.write_image(str(path), scale=2, width=width, height=height)
    print(f"💾 {path}")
    fig.show()

print(f"📁 Run-Ordner: {RUN_DIR}")

📁 Run-Ordner: /Users/andrey/Repositories/fs-thesis/models/runs/tab_pfn_20260305_220349


# 1. Data Pipeline Overview
This notebook uses the centralized data pipeline defined in `fs_thesis.data_loader`. Below is a documentation of the logic encapsulated in `load_final_data()`.

### A. Patient Demographics (Baseline)
We extract the following baseline features from the first admission (`t0_time`):
- **Identifier**: `subject_id`
- **Demographics**: `gender`, `anchor_age`, `race`, `marital_status`, `language`, `insurance`
- **Context**: `admission_type`
- **BMI**: Median BMI from `hosp.omr` (Left Join, missing values are handled by TabPFN)

### B. Event Definition (Target)
The event is defined as the **first occurrence** of a specific ICD diagnosis (e.g., Heart Failure `I50%`).
- **Event Time**: Timestamp of the diagnosis.
- **Censoring**: If no event occurs, the patient is censored at the date of death (`dod`) or end of follow-up.

### C. Target Calculation Logic
The target variable is derived based on the time-to-event (`duration`):
1. **Calculate Duration**:
   - `t_event` = Days from baseline to diagnosis.
   - `t_death` = Days from baseline to death.
   - Priority: Event Time > Death Time > Fallback (2000 days).
   - Negative durations are clipped to 0.

2. **Define Classes (`target`)**:
   - **Class 0 (Early Event)**: Event occurs $\le$ 365 days.
   - **Class 1 (Late Event)**: Event occurs $>$ 365 days.
   - **Class 2 (Censored/Control)**: No event observed (censored or healthy).

In [3]:
from fs_thesis.data_loader import load_final_data
df = load_final_data()

In [4]:
# Prüfen wie viele Missings wir haben
print("Missing BMI from Join, before feature engineering: TabPFN will handle these missing values, but it's good to know how many we have.")
print(f"Missing BMI: {df['bmi'].null_count()} of {len(df)}")

Missing BMI from Join, before feature engineering: TabPFN will handle these missing values, but it's good to know how many we have.
Missing BMI: 98306 of 223452


# 2. Preprocessing
## Splitting

In [5]:
from fs_thesis.preprocessing import preprocess_data, balance_data, get_X_y
df_train, df_val, df_test = preprocess_data(df)

Shapes -> Train: (143008, 17), Val: (35753, 17), Test: (44691, 17)


## Sampling (Balancing)

In [6]:
# Balance (only for train data!)
df_balanced = balance_data(df_train, n_samples=500) # definiert wie lange es läuft range (100 - 1000) 500 plateaut 

# split Features & Target (all Sets!)
X_train, y_train = get_X_y(df_balanced)
X_val, y_val = get_X_y(df_val)
X_test, y_test = get_X_y(df_test)

# Jetzt passt auch der fucking Print
print(f"Train (balanced): {len(y_train)} | Val (real): {len(y_val)} | Test (real): {len(y_test)}")

Train (balanced): 1500 | Val (real): 35753 | Test (real): 44691


# 3. Training
## Classifier

In [7]:
from tabpfn import TabPFNClassifier
classifier = TabPFNClassifier(device='mps') # Zurück auf CPU, für schnellere Vorhersagen bei kleinen N

## Fit

In [8]:
print("Start fitting...")
classifier.fit(X_train, y_train)
print("Training done!")

Start fitting...
Training done!


## Predict

In [9]:
# 3. Vorhersage (Validierung)

# For importance analysis
skip = False

print("Start predict...")
y_val_pred = classifier.predict(X_val)
print("Done!")

Start predict...
Done!


# 4. Validation (val_set for optimizing)

In [10]:

print("Evaluating...")
y_proba = classifier.predict_proba(X_val)
roc_auc = roc_auc_score(y_val, y_proba, multi_class='ovr', average='macro')

Evaluating...


In [11]:
print(f"Accuracy: {accuracy_score(y_val, y_val_pred):.2f}")
print(f"AUC - ROC Score: {roc_auc:.2f}")

print("\nClassification Report:")
print(classification_report(y_val, y_val_pred, 
                            target_names=['early (<1J)', 'late (>1J)', 'healthy']))

Accuracy: 0.54
AUC - ROC Score: 0.81

Classification Report:
              precision    recall  f1-score   support

 early (<1J)       0.13      0.67      0.21      1719
  late (>1J)       0.10      0.73      0.18      1304
     healthy       0.98      0.53      0.69     32730

    accuracy                           0.54     35753
   macro avg       0.40      0.64      0.36     35753
weighted avg       0.91      0.54      0.65     35753



In [12]:
# ROC-AUC Visualisierung (Validation)
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc

y_val_bin = label_binarize(y_val, classes=[0, 1, 2])
class_names = ['Früh (<1J)', 'Spät (>1J)', 'Gesund']
colors = ['#e74c3c', '#e67e22', '#2ecc71']

fig = go.Figure()

for i, (name, color) in enumerate(zip(class_names, colors)):
    fpr, tpr, _ = roc_curve(y_val_bin[:, i], y_proba[:, i])
    roc_auc_val = auc(fpr, tpr)
    fig.add_trace(go.Scatter(
        x=fpr, y=tpr, mode='lines',
        name=f'{name} (AUC={roc_auc_val:.3f})',
        line=dict(color=color, width=2)
    ))

fig.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1], mode='lines',
    name='Zufall (AUC=0.500)',
    line=dict(color='grey', width=1, dash='dash')
))

fig.update_layout(
    title=f'ROC Curve (One-vs-Rest) — Validation Set (n={len(y_val)})',
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate (Sensitivity)',
    template='plotly_white',
    legend=dict(x=0.55, y=0.05),
    width=800, height=700
)

show_and_save(fig, "roc_auc_validation")

💾 /Users/andrey/Repositories/fs-thesis/models/runs/tab_pfn_20260305_220349/plots/roc_auc_validation.png


# 4.1 Visualization Data

In [13]:
import plotly.express as px

df_analyze = X_val.copy()
if hasattr(df_analyze, "to_pandas"):
    df_analyze = df_analyze.to_pandas()

fig = px.box(df_analyze, x="insurance", y="anchor_age", 
             title="'Medicare'-Effekt, be causion with age for TabPFN",
             points="all", 
             color="insurance")

show_and_save(fig, "medicare_age_effect")

💾 /Users/andrey/Repositories/fs-thesis/models/runs/tab_pfn_20260305_220349/plots/medicare_age_effect.png


In [14]:
risk_score = y_proba[:, 0]  # ← Klasse 0 = early (Risiko-Score)

df_analyze['risk_score'] = risk_score
df_analyze['true_label'] = y_val

# 4.2 Visualisation Prediction Performance

In [ ]:
cm = confusion_matrix(y_val, y_val_pred)
cm_perc = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

labels = ['early (<1J)', 'late (1-3J)', 'healthy']

annot_text = [
    [f"<b>{val}</b><br>({perc:.1%})" for val, perc in zip(row_val, row_perc)]
    for row_val, row_perc in zip(cm, cm_perc)
]

fig = ff.create_annotated_heatmap(
    cm_perc, 
    x=labels, 
    y=labels, 
    annotation_text=annot_text, 
    colorscale='Reds'
)
fig.update_layout(
    title=f'Validation Check: Confusion Matrix ({len(y_val)} Samples)',
    xaxis_title="Vorhersage des Modells",
    yaxis_title="Tatsächlicher Verlauf (MIMIC-Daten)",
    template="plotly_white",
    height=600
)

show_and_save(fig, "confusion_matrix")

💾 /Users/andrey/Repositories/fs-thesis/models/runs/tab_pfn_20260305_214544/plots/confusion_matrix.png


In [ ]:
import plotly.graph_objects as go

cm = confusion_matrix(y_val, y_val_pred)

label_list = [
    "real: early", "real: late", "real: healthy",
    "predicted: early", "predicted: late", "predicted: healthy"
]

source = [0, 0, 0, 1, 1, 1, 2, 2, 2]
target = [3, 4, 5, 3, 4, 5, 3, 4, 5]
value = cm.flatten()

color_link = [
    'rgba(255, 90, 90, 0.4)', 'rgba(255, 90, 90, 0.2)', 'rgba(255, 90, 90, 0.1)',
    'rgba(255, 127, 14, 0.2)', 'rgba(255, 127, 14, 0.4)', 'rgba(255, 127, 14, 0.1)',
    'rgba(44, 160, 44, 0.1)', 'rgba(44, 160, 44, 0.1)', 'rgba(44, 160, 44, 0.4)'
]

fig2 = go.Figure(data=[go.Sankey(
    node = dict(
      pad = 15, thickness = 20, line = dict(color = "black", width = 0.5),
      label = label_list, color = "grey"
    ),
    link = dict(
      source = source, target = target, value = value, color = color_link
  ))])

fig2.update_layout(title_text="Patient-Flow: Reality vs. Predicted", font_size=12)
show_and_save(fig2, "sankey_patient_flow")

💾 /Users/andrey/Repositories/fs-thesis/models/runs/tab_pfn_20260305_214544/plots/sankey_patient_flow.png


# 6 Robustness Check

In [ ]:
# --- Configuration ---
N_LOOPS = 20
RUN_NAME = "robustness_overnight"

import warnings, time, json
from tqdm.auto import tqdm
from sklearn.metrics import recall_score, precision_score

warnings.filterwarnings('ignore', message='.*Running on CPU with more than 200 samples.*')

log_file = RUN_DIR / "run.log"

def log(msg):
    print(msg)
    with open(log_file, 'a') as f:
        f.write(f"{datetime.now().strftime('%H:%M:%S')} | {msg}\n")

configs = [
    {"name": "ensemble_8",  "N_ensemble": 8,  "n_samples": 300},
    # {"name": "ensemble_32", "N_ensemble": 32,  "n_samples": 300}, --- IGNORE --- no difference to 8, but much slower
]

# Config speichern
json.dump({"n_loops": N_LOOPS, "configs": configs, "run_dir": str(RUN_DIR)},
          open(RUN_DIR / "config.json", "w"), indent=2)

log(f"RUN: {RUN_DIR.name} | Loops: {N_LOOPS} | Configs: {len(configs)}")

results = []
start_total = time.time()

for cfg_idx, config in enumerate(configs, 1):
    t0 = time.time()
    log(f"\n[{cfg_idx}/{len(configs)}] {config['name']} | ensemble={config['N_ensemble']}, samples={config['n_samples']}")
    config_results = []
    
    for i in tqdm(range(N_LOOPS), desc=f"[{cfg_idx}/{len(configs)}] {config['name']}"):
        try:
            try:
                clf = TabPFNClassifier(device='mps', n_estimators=config['N_ensemble'])
            except TypeError:
                clf = TabPFNClassifier(device='mps')
            
            df_bal = balance_data(df_train, n_samples=config['n_samples'], seed=42 + i)
            X_tr, y_tr = get_X_y(df_bal)
            clf.fit(X_tr, y_tr)
            y_pred = clf.predict(X_val)
            y_proba = clf.predict_proba(X_val)
            f1_pc = f1_score(y_val, y_pred, average=None)
            
            result = {
                'run_id': i, 'config_name': config['name'],
                'n_ensemble': config['N_ensemble'], 'n_samples': config['n_samples'],
                'accuracy': accuracy_score(y_val, y_pred),
                'roc_auc_macro': roc_auc_score(y_val, y_proba, multi_class='ovr', average='macro'),
                'f1_macro': f1_score(y_val, y_pred, average='macro'),
                'recall_macro': recall_score(y_val, y_pred, average='macro'),
                'precision_macro': precision_score(y_val, y_pred, average='macro'),
                'f1_class_0_early': f1_pc[0],
                'f1_class_1_late': f1_pc[1],
                'f1_class_2_healthy': f1_pc[2],
                'seed': 42 + i,
            }
            results.append(result)
            config_results.append(result)
        except Exception as e:
            log(f"  ⚠️ FEHLER Run {i}: {e}")

    elapsed = time.time() - t0
    if config_results:
        df_cfg = pd.DataFrame(config_results)
        log(f"  ✅ F1={df_cfg['f1_macro'].mean():.4f}±{df_cfg['f1_macro'].std():.4f} | {elapsed/60:.1f}min")
        pd.DataFrame(results).to_csv(RESULTS_DIR / f"checkpoint_config{cfg_idx}.csv", index=False)


RUN: tab_pfn_20260305_214544 | Loops: 20 | Configs: 1

[1/1] ensemble_8 | ensemble=8, samples=300


[1/1] ensemble_8:   0%|          | 0/20 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:

# Finale Ergebnisse
df_results = pd.DataFrame(results)
df_results.to_csv(RESULTS_DIR / "final_results.csv", index=False)
log(f"\nDONE in {(time.time()-start_total)/3600:.2f}h | {len(df_results)} runs")

# Summary Plots
df_summary = df_results.groupby('config_name').agg(
    f1_mean=('f1_macro','mean'), f1_std=('f1_macro','std'),
).reset_index().sort_values('f1_mean', ascending=True)

fig = px.bar(df_summary, x='f1_mean', y='config_name', error_x='f1_std',
    orientation='h', title='F1 Macro: Config Comparison', template='plotly_white', text_auto='.3f')
fig.update_layout(xaxis_tickformat='.0%', xaxis_title='F1 Macro', yaxis_title=None)
show_and_save(fig, "robustness_config_comparison")

fig2 = px.violin(df_results, x='config_name', y='f1_macro', box=True, points='all',
    title='F1 Macro Distribution per Config', template='plotly_white')
fig2.update_layout(yaxis_tickformat='.0%', xaxis_tickangle=45)
show_and_save(fig2, "robustness_f1_violin")

# Summary Table
df_summary_full = df_results.groupby('config_name').agg(
    f1_mean=('f1_macro','mean'), f1_std=('f1_macro','std'),
    acc_mean=('accuracy','mean'), acc_std=('accuracy','std'),
    auc_mean=('roc_auc_macro','mean'), auc_std=('roc_auc_macro','std'),
    f1_early_mean=('f1_class_0_early','mean'), f1_early_std=('f1_class_0_early','std'),
    f1_late_mean=('f1_class_1_late','mean'), f1_late_std=('f1_class_1_late','std'),
    f1_healthy_mean=('f1_class_2_healthy','mean'), f1_healthy_std=('f1_class_2_healthy','std'),
    n_runs=('run_id','count'),
).reset_index()
df_summary_full.to_csv(RESULTS_DIR / "summary_table.csv", index=False)

log(f"\n📁 {RUN_DIR} | plots: {len(list(PLOTS_DIR.glob('*.png')))} | results: {len(list(RESULTS_DIR.glob('*.csv')))}")


DONE in 1.20h | 20 runs
💾 /Users/andrey/Repositories/fs-thesis/models/runs/tab_pfn_20260304_194933/plots/robustness_config_comparison.png


💾 /Users/andrey/Repositories/fs-thesis/models/runs/tab_pfn_20260304_194933/plots/robustness_f1_violin.png



📁 /Users/andrey/Repositories/fs-thesis/models/runs/tab_pfn_20260304_194933 | plots: 5 | results: 3


# 6.1 Robustness Vizualiszation

In [ ]:
# 4. Visualisierung der Robustness (Master-Thesis Style)
import plotly.express as px
import os

df_melt = df_results.melt(
    id_vars=['run_id'], 
    value_vars=['f1_class_0_early', 'f1_class_1_late', 'f1_class_2_healthy'],
    var_name='Target Class', 
    value_name='F1 Score'
)

df_stats = df_melt.groupby('Target Class')['F1 Score'].agg(['mean', 'std']).reset_index()

category_order = ['Früher Ausbruch (<1J)', 'Später Ausbruch (1-3J)', 'Gesund / Kein Event']

name_mapping = {
    'f1_class_0_early': 'Früher Ausbruch (<1J)',
    'f1_class_1_late': 'Später Ausbruch (1-3J)',
    'f1_class_2_healthy': 'Gesund / Kein Event'
}
df_stats['Target Class'] = df_stats['Target Class'].map(name_mapping)
df_melt['Target Class'] = df_melt['Target Class'].map(name_mapping)

fig = px.bar(
    df_stats, 
    x="Target Class", 
    y="mean", 
    error_y="std",
    title=f"Modell-Performance: Durchschnitt & Stabilität ({N_LOOPS} Runs)",
    text_auto='.1%',
    labels={'mean': 'Durchschnittlicher F1-Score'},
    color="Target Class", 
    color_discrete_sequence=px.colors.qualitative.Pastel,
    template="plotly_white",
    category_orders={"Target Class": category_order}
)

fig.update_traces(marker_opacity=0.8, showlegend=False)

scatter_trace = px.strip(
    df_melt, 
    x="Target Class", 
    y="F1 Score", 
    category_orders={"Target Class": category_order}
).data[0]

scatter_trace.marker.color = '#34495e'
scatter_trace.marker.size = 6
scatter_trace.marker.opacity = 0.8
scatter_trace.marker.line = dict(width=1, color='white')
scatter_trace.showlegend = False

fig.add_trace(scatter_trace)

fig.update_layout(
    yaxis_tickformat='.0%', 
    yaxis_title="F1 Score (Macro)", 
    xaxis_title=None,
    font=dict(size=14)
)
fig.update_yaxes(range=[0, 1.1])

show_and_save(fig, "robustness_f1_per_class")

💾 /Users/andrey/Repositories/fs-thesis/models/runs/tab_pfn_20260304_194933/plots/robustness_f1_per_class.png


# 6.2 Test

In [ ]:
# Nach dem Robustness Loop: Bestes Modell auf TEST-Set evaluieren
best_seed = df_results.loc[df_results['f1_macro'].idxmax(), 'seed']
best_config = df_results.loc[df_results['f1_macro'].idxmax(), 'config_name']
log(f"Best config: {best_config}, seed: {int(best_seed)}")

# Reproduziere das beste Modell
df_bal_best = balance_data(df_train, n_samples=300, seed=int(best_seed))
X_tr_best, y_tr_best = get_X_y(df_bal_best)
clf_best = TabPFNClassifier(device='mps')
clf_best.fit(X_tr_best, y_tr_best)

# Finale Evaluation auf TEST (nicht Val!)
y_test_pred_final = clf_best.predict(X_test)
y_test_proba_final = clf_best.predict_proba(X_test)

test_metrics = {
    'accuracy': accuracy_score(y_test, y_test_pred_final),
    'f1_macro': f1_score(y_test, y_test_pred_final, average='macro'),
    'roc_auc': roc_auc_score(y_test, y_test_proba_final, multi_class='ovr', average='macro'),
}
pd.DataFrame([test_metrics]).to_csv(RESULTS_DIR / "test_final_metrics.csv", index=False)

print(f"📊 FINAL TEST: Acc={test_metrics['accuracy']:.2%} | F1={test_metrics['f1_macro']:.2%} | AUC={test_metrics['roc_auc']:.2%}")
print(classification_report(y_test, y_test_pred_final, 
                            target_names=['Früh (<1J)', 'Spät (>1J)', 'Gesund']))

Best config: ensemble_8, seed: 46
📊 FINAL TEST: Acc=59.34% | F1=38.52% | AUC=80.48%
              precision    recall  f1-score   support

  Früh (<1J)       0.15      0.59      0.24      2148
  Spät (>1J)       0.10      0.74      0.18      1630
      Gesund       0.98      0.59      0.73     40913

    accuracy                           0.59     44691
   macro avg       0.41      0.64      0.39     44691
weighted avg       0.90      0.59      0.69     44691



### 6.3 Risk-Analyse

In [ ]:

MIN_GROUP_SIZE = 50

y_proba_best = clf_best.predict_proba(X_val)
risk_score = y_proba_best[:, 0]  # ← Klasse 0 = early (Risiko-Score)

df_analyze = X_val.copy()
if hasattr(df_analyze, "to_pandas"):
    df_analyze = df_analyze.to_pandas()

df_analyze['risk_score'] = risk_score
df_analyze['true_label'] = y_val

def risk_by_feature(df, feature, title, fig_name, angle=0):
    stats = df.groupby(feature).agg(
        risk_mean=('risk_score', 'mean'),
        risk_std=('risk_score', 'std'),
        n=('risk_score', 'count')
    ).reset_index()
    
    excluded = stats[stats['n'] < MIN_GROUP_SIZE]
    stats = stats[stats['n'] >= MIN_GROUP_SIZE].sort_values('risk_mean', ascending=True)
    
    fig = px.bar(
        stats, x=feature, y='risk_mean', error_y='risk_std',
        text=stats.apply(lambda r: f"{r['risk_mean']:.1%}<br>(n={int(r['n'])})", axis=1),
        title=f'{title} (min n={MIN_GROUP_SIZE})',
        labels={'risk_mean': 'Risk Score'},
        color='risk_mean', color_continuous_scale='Reds', template="plotly_white"
    )
    fig.update_layout(yaxis_tickformat='.0%', xaxis_tickangle=angle)
    
    if not excluded.empty:
        excluded_str = ", ".join(f"{r[feature]} (n={int(r['n'])})" for _, r in excluded.iterrows())
        print(f"  ⚠️ Excluded (n<{MIN_GROUP_SIZE}): {excluded_str}")
    
    show_and_save(fig, fig_name)

In [ ]:


# BMI braucht Sonderbehandlung (Binning)
bins = [0, 18.5, 25, 30, 100]
labels = ['Untergewicht (<18.5)', 'Normal (18.5-25)', 'Übergewicht (25-30)', 'Adipositas (>30)']
df_analyze['bmi_group'] = pd.cut(df_analyze['bmi'], bins=bins, labels=labels)
print(f"  ℹ️ BMI missing: {df_analyze['bmi_group'].isna().sum()} patients excluded from BMI plot")

# Age braucht Sonderbehandlung (Binning)
df_analyze['age_group'] = pd.cut(
    df_analyze['anchor_age'],
    bins=[0, 20, 30, 40, 50, 60, 70, 80, 90, 120],
    labels=['<20', '20-29', '30-39', '40-49', '50-59', '60-69', '70-79', '80-89', '90+']
).astype(str)

# Alle Plots
risk_by_feature(df_analyze, 'bmi_group', 'Avg Risk by BMI Group', 'risk_by_bmi')
risk_by_feature(df_analyze, 'age_group', 'Avg Risk by Age Group', 'risk_by_age')
risk_by_feature(df_analyze, 'gender', 'Avg Risk by Gender', 'risk_by_gender')
risk_by_feature(df_analyze, 'insurance', 'Avg Risk by Insurance', 'risk_by_insurance')
risk_by_feature(df_analyze, 'language', 'Avg Risk by Language', 'risk_by_language')
risk_by_feature(df_analyze, 'marital_status', 'Avg Risk by Marital Status', 'risk_by_marital_status')
risk_by_feature(df_analyze, 'race', 'Avg Risk by Race', 'risk_by_race', angle=45)
risk_by_feature(df_analyze, 'admission_type', 'Avg Risk by Admission Type', 'risk_by_admission_type', angle=45)

  ℹ️ BMI missing: 15858 patients excluded from BMI plot


/var/folders/z6/l14kg6kd267dqw00p40_86jc0000gn/T/ipykernel_57054/485537832.py:14: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



💾 /Users/andrey/Repositories/fs-thesis/models/runs/tab_pfn_20260304_194933/plots/risk_by_bmi.png


💾 /Users/andrey/Repositories/fs-thesis/models/runs/tab_pfn_20260304_194933/plots/risk_by_age.png


💾 /Users/andrey/Repositories/fs-thesis/models/runs/tab_pfn_20260304_194933/plots/risk_by_gender.png


  ⚠️ Excluded (n<50): No charge (n=29)
💾 /Users/andrey/Repositories/fs-thesis/models/runs/tab_pfn_20260304_194933/plots/risk_by_insurance.png


  ⚠️ Excluded (n<50): American Sign Language (n=13), Amharic (n=21), Arabic (n=41), Armenian (n=11), Bengali (n=9), French (n=13), Hindi (n=17), Japanese (n=18), Khmer (n=18), Korean (n=25), Persian (n=23), Polish (n=21), Somali (n=10), Thai (n=17)
💾 /Users/andrey/Repositories/fs-thesis/models/runs/tab_pfn_20260304_194933/plots/risk_by_language.png


💾 /Users/andrey/Repositories/fs-thesis/models/runs/tab_pfn_20260304_194933/plots/risk_by_marital_status.png


  ⚠️ Excluded (n<50): ASIAN - KOREAN (n=37), HISPANIC/LATINO - CENTRAL AMERICAN (n=28), HISPANIC/LATINO - CUBAN (n=30), HISPANIC/LATINO - HONDURAN (n=41), NATIVE HAWAIIAN OR OTHER PACIFIC ISLANDER (n=40)
💾 /Users/andrey/Repositories/fs-thesis/models/runs/tab_pfn_20260304_194933/plots/risk_by_race.png


💾 /Users/andrey/Repositories/fs-thesis/models/runs/tab_pfn_20260304_194933/plots/risk_by_admission_type.png


# 7. Feature Importance

In [ ]:
from sklearn.inspection import permutation_importance
import pandas as pd
import plotly.express as px

if skip:
    print("skipped, dauert zu lang der Scheiss")
else:
    # Bestes Modell aus Robustness Loop verwenden (nicht den alten classifier!)
    print("Berechne Feature Importance mit bestem Modell (n_repeats=10)...")

    result = permutation_importance(
        clf_best,          # ← bestes Modell aus Section 6. Test
        X_val,             # ← volles Val-Set, nicht Sample!
        y_val,
        n_repeats=10,      # ← Standard für Thesis
        random_state=42, 
        scoring='f1_macro',  # ← passend zu deiner Hauptmetrik
        n_jobs=1
    )

    importance_df = pd.DataFrame({
        'Feature': X_val.columns,  # ← dynamisch statt hardcoded
        'Importance': result.importances_mean,
        'Std_Dev': result.importances_std
    }).sort_values(by='Importance', ascending=True)
    
    # Speichern für Thesis
    importance_df.to_csv(RESULTS_DIR / "feature_importance.csv", index=False)

    fig = px.bar(
        importance_df, 
        x='Importance', 
        y='Feature', 
        orientation='h',
        title='Feature Importance (Permutation, n=10)',
        labels={'Importance': 'Mean Decrease in F1 Macro'},
        error_x='Std_Dev',
        template="plotly_white",
        color='Importance',
        color_continuous_scale='Reds'
    )

    fig.update_layout(height=500, xaxis_tickformat='.1%')
    show_and_save(fig, "feature_importance")

Berechne Feature Importance mit bestem Modell (n_repeats=10)...
💾 /Users/andrey/Repositories/fs-thesis/models/runs/tab_pfn_20260304_194933/plots/feature_importance.png


# Philipp Feedback
Batehosenträger rausnehmen. Also die eine Korrelation haben aber keine Kausalität.
Interpretieren, was eine Kausalität ist. 

Kommt Bullshit raus habe ich keine relevanten Features. (also keine Kausalität)
Gehts von der Kausaöit#t zur Ursache? Falls es nicht geht, dann reicht einfach Korrelation. 


AOC (Area in a Curve)
Sensivity einbauen  n.   